<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_gated_karpathy_sts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT GatedGPT — Karpathy-sized (Shakespeare + TinyStories char)

Single transformer with per-neuron mask-based specialty routing. Each MLP inner unit (4*n_embd) and each attention head gets a fixed mask value m sampled at init from Beta(0.5, 0.5) (arcsine — tail-heavy, ~20% near 0, ~20% near 1, the rest in middle). At forward, a scalar alpha gates each unit by `alpha*m + (1-alpha)*(1-m)`, so m=1 specialists fire at alpha=1, m=0 specialists fire at alpha=0, and m=0.5 halfsies fire at 0.5 regardless.

**Per-slot architecture matches Karpathy's `train_shakespeare_char.py` exactly** (n_layer=6, n_head=6, n_embd=384, dropout=0.2, lr=1e-3). Total params ~10.65M, same as Karpathy. Effective capacity at alpha=1 is roughly half (only m_i units fire); to match Karpathy's full reference run the medium-sized notebook instead.

Iter budget: 10000 (vs Karpathy's 5000 single-model). Doubled because under Bernoulli(alpha) corpus selection, each specialty group sees roughly half the iters.

T4 runtime estimate: ~90 minutes.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT
if '/content/abcGPT' not in sys.path:
    sys.path.insert(0, '/content/abcGPT')

In [ ]:
!python data/shakespeare_tinystories_char/prepare.py

## Data: per-corpus batch loaders

In [ ]:
import os, pickle
import numpy as np
import torch

data_dir = '/content/abcGPT/data/shakespeare_tinystories_char'
train_arr = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
with open(os.path.join(data_dir, 'meta.pkl'), 'rb') as f:
    meta = pickle.load(f)
vocab_size = meta['vocab_size']
stoi, itos = meta['stoi'], meta['itos']

# Find the separator '\n\n===\n\n' to split train.bin into shake / ts ranges
sep_str = '\n\n===\n\n'
sep_ids = np.array([stoi[c] for c in sep_str], dtype=np.uint16)
candidates = np.where(train_arr[:len(train_arr) - len(sep_ids) + 1] == sep_ids[0])[0]
sep_idx = None
for c in candidates:
    if np.array_equal(train_arr[c:c+len(sep_ids)], sep_ids):
        sep_idx = int(c); break
shake_end = sep_idx
ts_start = sep_idx + len(sep_ids)
ts_end = len(train_arr)
print(f'vocab_size={vocab_size}  shake=[0:{shake_end}] ts=[{ts_start}:{ts_end}]')

In [ ]:
block_size = 256
batch_size = 64
device = 'cuda'

def make_get_batch(arr, start, end):
    span = end - start
    def get_batch():
        ix = torch.randint(span - block_size, (batch_size,)) + start
        x = torch.stack([torch.from_numpy(arr[i:i+block_size].astype(np.int64)) for i in ix.tolist()])
        y = torch.stack([torch.from_numpy(arr[i+1:i+1+block_size].astype(np.int64)) for i in ix.tolist()])
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)
    return get_batch

shake_get_batch = make_get_batch(train_arr, 0, shake_end)
ts_get_batch    = make_get_batch(train_arr, ts_start, ts_end)

def val_range(start, end, frac=0.1):
    span = end - start
    return end - int(span * frac), end
shake_vs, shake_ve = val_range(0, shake_end)
ts_vs,    ts_ve    = val_range(ts_start, ts_end)
shake_val_batch = make_get_batch(train_arr, shake_vs, shake_ve)
ts_val_batch    = make_get_batch(train_arr, ts_vs,    ts_ve)

## Build GatedGPT (Karpathy-sized)

Same architecture as Karpathy's char-shake (n_layer=6, n_head=6, n_embd=384, ~10.65M params). The masks are sampled at init with `mask_seed=1337` so re-runs of this notebook produce the same specialty assignment.

In [ ]:
from gated_gpt import GatedGPT, GatedGPTConfig, train_gated

config = GatedGPTConfig(
    vocab_size=vocab_size,
    n_layer=6, n_head=6, n_embd=384,
    block_size=block_size,
    dropout=0.2,
    bias=False,
    mask_seed=1337,
)
torch.manual_seed(1337)
model = GatedGPT(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'GatedGPT initialized: {n_params/1e6:.2f}M params (Karpathy reference is 10.65M)')

## Mask summary

Sanity check: under Beta(0.5, 0.5) we expect ~20% of units below 0.1 (ts specialists), ~20% above 0.9 (shake specialists), and ~60% in the middle (halfsies).

In [ ]:
for k, v in model.mask_summary().items():
    print(f'{k}: mean={v["mean"]:.3f}  shake-spec={v["shake_specialist (m>0.9)"]}/{v["total"]}  '
          f'ts-spec={v["ts_specialist (m<0.1)"]}/{v["total"]}  halfsies={v["halfsies (0.1..0.9)"]}/{v["total"]}')

import matplotlib.pyplot as plt
all_m = torch.cat([b.mlp.M for b in model.transformer.h] + [b.attn.M for b in model.transformer.h]).cpu().numpy()
plt.figure(figsize=(6, 3))
plt.hist(all_m, bins=40, edgecolor='black')
plt.title(f'mask histogram across all {len(all_m)} gated units')
plt.xlabel('m'); plt.ylabel('count'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Train

Per iter: sample alpha from Beta(0.5,0.5), then choose corpus by Bernoulli(alpha) (high alpha favors shake). At alpha=1 the iter trains shake specialists on shake data; at alpha=0 it trains ts specialists on ts data; mid-alpha lands halfsies stochastically on whichever corpus came up. The val printouts at corners show the effective per-corpus loss the user will see when they drag the slider to that end.

In [ ]:
train_gated(
    model, shake_get_batch, ts_get_batch,
    n_iters=10000, lr=1e-3, warmup=100, lr_decay_iters=10000, min_lr=1e-4,
    beta2=0.99, weight_decay=0.1, grad_clip=1.0,
    alpha_dist='beta_half',
    log_interval=250, eval_interval=500, eval_iters=200,
    get_shake_val=shake_val_batch, get_ts_val=ts_val_batch,
    device=device, amp_dtype=torch.bfloat16,
)

In [ ]:
ckpt_dir = '/content/drive/MyDrive/abcGPT/gated_karpathy_sts'
os.makedirs(ckpt_dir, exist_ok=True)
torch.save({'model_state': model.state_dict(), 'config': config.__dict__}, f'{ckpt_dir}/model.pt')
print(f'saved to {ckpt_dir}/model.pt')

## Slider sweep: val loss across alpha

The model is one set of weights. We just vary alpha at forward time and re-evaluate. If the gating worked, alpha=1 should hit a shake-specialist's local minimum on shake and alpha=0 should hit a ts-specialist's on ts.

In [ ]:
@torch.no_grad()
def eval_loss(alpha, get_batch_fn, n_iters=50):
    model.eval()
    losses = []
    for _ in range(n_iters):
        X, Y = get_batch_fn()
        with torch.amp.autocast(device_type=device, dtype=torch.bfloat16):
            _, loss = model(X, alpha, Y)
        losses.append(loss.item())
    return float(np.mean(losses))

alphas = np.linspace(0.0, 1.0, 11)
rows = []
for a in alphas:
    sh = eval_loss(float(a), shake_val_batch)
    ts = eval_loss(float(a), ts_val_batch)
    rows.append((a, sh, ts))
    print(f'alpha={a:.2f}  shake_val={sh:.3f}  ts_val={ts:.3f}')
rows = np.array(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rows[:,0], rows[:,1], 'o-')
axes[0].set_xlabel('alpha (1.0 = shake mode)'); axes[0].set_ylabel('val loss on tinyshakespeare')
axes[0].set_title('shake val'); axes[0].grid(alpha=0.3)
axes[1].plot(rows[:,0], rows[:,2], 'o-')
axes[1].set_xlabel('alpha (1.0 = shake mode)'); axes[1].set_ylabel('val loss on TinyStories')
axes[1].set_title('tinystories val'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Sample text across alpha

In [ ]:
def encode(s):
    return [stoi[c] for c in s if c in stoi]
def decode(ids):
    return ''.join(itos[i] for i in ids)

prompt = 'the king of '
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    torch.manual_seed(0)
    out = model.generate(prompt_ids, alpha=float(a), max_new_tokens=200, temperature=0.8, top_k=40)
    print('=' * 70)
    print(f'alpha = {a}')
    print(decode(out[0].tolist()))